# 02613 Exam 2024 — Code Proofs
Each cell proves the answer to one question by running the actual computation.


In [ ]:
import numpy as np


## Q2 — Parallel fraction from speed-up plot (saturates at 5)
Answer: **C) F = 0.8**


In [ ]:
S_max = 5  # plot saturates at 5
F = 1 - 1/S_max
print(f'F = {F}')  # should print 0.8

# Verify all options:
for F_option in [0.2, 0.5, 0.8, 0.9]:
    s_max = 1 / (1 - F_option)
    print(f'F={F_option} → S_max={s_max:.2f}')


## Q3 — Should they parallelise? (F=0.8, need S≥4, max 8 cores)
Answer: **No — S(8) ≈ 3.33 < 4**


In [ ]:
def amdahl(F, p):
    return 1 / ((1 - F) + F/p)

F = 0.8
p = 8
S8 = amdahl(F, p)
print(f'S(8) = {S8:.4f}')         # ≈ 3.33
print(f'Target met: {S8 >= 4}')   # False — do NOT parallelise
print(f'S_max = {1/(1-F):.2f}')   # max possible = 5, still above 4 with infinite cores
# But with 8 cores they only get 3.33 — below threshold


## Q7 — Broadcasting: subtract mean image from images
images: (N,H,W,3), mim: (H,W) → need `mim[:,:,None]`
Answer: **B) `images - mim[:, :, None]`**


In [ ]:
N, H, W, C = 10, 64, 64, 3
images = np.zeros((N, H, W, C))
mim = np.zeros((H, W))

# Option A: images - mim  → should fail
try:
    r = images - mim
    print(f'A: shape {r.shape}')  # might work but wrong semantics
except Exception as e:
    print(f'A: ERROR — {e}')

# Option B: images - mim[:,:,None]  → correct
try:
    r = images - mim[:, :, None]
    print(f'B: shape {r.shape}')  # should be (10,64,64,3) ✓
except Exception as e:
    print(f'B: ERROR — {e}')

# Option C: images - mim[None]  → shape mismatch
try:
    r = images - mim[None]
    print(f'C: shape {r.shape}')  # may fail or give wrong shape
except Exception as e:
    print(f'C: ERROR — {e}')


## Q8 — Loop reordering from strides (600, 40, 8, 200)
Answer: innermost→outermost: **k, j, l, i**


In [ ]:
strides = {0: 600, 1: 40, 2: 8, 3: 200}
names   = {0: 'i', 1: 'j', 2: 'k', 3: 'l'}

# Sort axes by stride ascending → innermost first
sorted_axes = sorted(strides, key=lambda ax: strides[ax])
print('Inner → outer loop order:', [names[ax] for ax in sorted_axes])
# Expected: k(8), j(40), l(200), i(600)

# Create a real array and verify strides match
# strides depend on shape and dtype — build the shape that gives (600,40,8,200)
# stride bytes: axis3=200, axis2=8, means shape[3]=200/8=25 elements, dtype=8 bytes (float64)
# stride[2]=8 means it IS the last float64 element → last dim has stride 8
# Actually strides given are arbitrary — just verify sorting logic
print('Strides sorted:', sorted(strides.values()))
print('Loop order (inner→outer):', 'k, j, l, i')


## Q9 — Number of samples from profiler (ncalls of process_sample = 10)
Answer: **C) 10**


In [ ]:
# Profiler shows process_sample ncalls=10 → 10 samples
# Simple verification: if process_sample called once per sample, ncalls = n_samples
ncalls_process_sample = 10
n_samples = ncalls_process_sample
print(f'Number of samples: {n_samples}')  # 10


## Q10 — Which function to optimise for 1000 samples?
Answer: **process_sample** (scales with sample count)


In [ ]:
# From profiler (10-sample subset):
profile = {
    'load_params':    {'ncalls': 1,  'percall': 20.009, 'scales': False},
    'prepare_model':  {'ncalls': 1,  'percall': 15.013, 'scales': False},
    'process_sample': {'ncalls': 10, 'percall': 0.505,  'scales': True},
    'save':           {'ncalls': 1,  'percall': 3.007,  'scales': False},
    'load_data':      {'ncalls': 1,  'percall': 1.001,  'scales': False},
}

production_samples = 1000
print('Projected production time:')
for name, p in profile.items():
    if p['scales']:
        t = p['percall'] * production_samples
    else:
        t = p['percall']  # fixed cost
    print(f'  {name:20s}: {t:8.1f}s  (scales={p["scales"]})')

# process_sample = 0.505 * 1000 = 505s dominates everything else


## Q13 — GPU transfer speed from nsys (HtoD)
Answer: **B) ~10 GB/s**


In [ ]:
htod_size_MB = 25000
htod_time_s  = 2.5
speed_MBs = htod_size_MB / htod_time_s
speed_GBs = speed_MBs / 1000
print(f'HtoD bandwidth: {speed_MBs:.0f} MB/s = {speed_GBs:.1f} GB/s')  # 10 GB/s


## Q14 — GPU vs CPU speedup (total GPU time = HtoD + kernel + DtoH)
Answer: **2x faster**


In [ ]:
htod   = 2.5   # s
kernel = 0.5   # s
dtoh   = 0.5   # s
gpu_total = htod + kernel + dtoh
cpu_time  = 7.0

speedup = cpu_time / gpu_total
print(f'GPU total time: {gpu_total} s')
print(f'CPU time:       {cpu_time} s')
print(f'Speedup:        {speedup:.1f}x')  # 2.0x
print(f'Kernel alone:   {cpu_time/kernel:.0f}x faster (but transfers dominate!)')


## Q15 — DataFrame dtype memory reduction
date→datetime, location→category/uint8, mach_id→int16, units→uint32


In [ ]:
import numpy as np

# Verify dtype ranges fit the data
checks = [
    ('mach_id',  -1, 5730,  np.int16),    # int16 range: -32768 to 32767
    ('units',   932, 68837, np.int32),     # int32 range: ±2.1e9
    ('units',   932, 68837, np.uint16),    # uint16: 0-65535, also fits
]
for name, lo, hi, dtype in checks:
    info = np.iinfo(dtype)
    fits = (lo >= info.min) and (hi <= info.max)
    print(f'{name} [{lo},{hi}] fits in {dtype.__name__} ({info.min} to {info.max}): {fits}')

# Memory comparison for int64 vs int16
n_rows = 1_000_000
print(f'\nmach_id: int64={n_rows*8/1e6:.0f}MB → int16={n_rows*2/1e6:.0f}MB (saves {n_rows*6/1e6:.0f}MB)')


## Q17 — Maximum chunk size (uint32 + uint64 + float64, 200 MB)
Answer: **B) ~10,000,000 rows**


In [ ]:
bytes_per_row = 4 + 8 + 8   # uint32=4, uint64=8, float64=8
available_MB  = 200
available_bytes = available_MB * 10**6

max_rows = available_bytes // bytes_per_row
print(f'Bytes per row:  {bytes_per_row}')
print(f'Available:      {available_bytes:,} bytes')
print(f'Max rows:       {max_rows:,}')  # 10,000,000


## Q19 — Can simulate_single be parallelised with threading?
`@jit(nogil=True)` releases GIL → ThreadPool works. Speed-up depends on m (outer calls), not n (inner steps).


In [ ]:
# Demonstrate GIL release with nogil — using a simple Numba example
# (requires numba; skip if not installed)
try:
    from numba import jit
    from multiprocessing.pool import ThreadPool
    from time import perf_counter

    @jit(nopython=True, nogil=True)
    def simulate_single(n):
        x = 0.0
        for i in range(n):
            x += i * 0.001
        return x

    simulate_single(100)  # warmup

    n = 10_000_000
    m = 4  # independent calls

    t = perf_counter()
    [simulate_single(n) for _ in range(m)]
    serial_time = perf_counter() - t

    t = perf_counter()
    with ThreadPool(m) as pool:
        pool.map(simulate_single, [n]*m)
    parallel_time = perf_counter() - t

    print(f'Serial:   {serial_time:.3f}s')
    print(f'Parallel: {parallel_time:.3f}s')
    print(f'Speedup:  {serial_time/parallel_time:.2f}x  (expected ~{m}x with {m} threads)')
except ImportError:
    print('numba not installed — install with: pip install numba')
